# End-to-End Asset Embeddings Pipeline

This notebook walks through the project's canonical training chain — **W2V → BERT-PT → BERT-FT** — on synthetic data, then evaluates all three resulting asset embeddings against a relative-valuation (RV) benchmark.

The configurations are tiny (sized for CPU), but the wiring is identical to production runs on real CSMAR data. End-to-end runtime is roughly one minute.

```text
   ┌─────────────────────────┐                     ┌─────────────────────────┐
   │ portfolio_pretrained.csv│                     │ portfolio_finetune.csv  │
   │      (PT period)        │                     │      (FT period)        │
   └────────────┬────────────┘                     └────────────┬────────────┘
                │                                               │
                ▼                                               │
       ┌─────────────────┐                                      │
       │  Step 1 · W2V   │                                      │
       │  d=16, 10 ep    │                                      │
       └────────┬────────┘                                      │
                │  init                                         │
                ▼                                               │
       ┌─────────────────┐                                      │
       │ Step 2 · BERT-PT│                                      │
       │ 2 layers, 8 ep  │                                      │
       └────────┬────────┘                                      │
                │  init                                         │
                ▼                                               │
       ┌─────────────────┐                                      │
       │ Step 3 · BERT-FT│ ◀────────────────────────────────────┘
       │   5 epochs      │
       └────────┬────────┘
                │
                │   Each step writes a *_embedding.csv;
                │   Step 4 reads all three.
                ▼
   ┌────────────────────────────────────────────────┐
   │  Step 4 · Relative Valuation × 3               │ ◀── benchmark.csv (2024Q2)
   ├────────────────────────────────────────────────┤
   │    W2V        RV ≈ 0.53     baseline           │
   │    BERT-PT    RV ≈ 0.53     ≈ W2V              │
   │    BERT-FT    RV ≈ 0.60     +0.07   ← FT lift  │
   └────────────────────────────────────────────────┘
```

**Reading the diagram.** BERT-PT inherits W2V's vocabulary and initial embedding matrix; BERT-FT warm-starts from BERT-PT and trains on a later-period portfolio set. The benchmark is generated from FT-period factors, so PT-trained models (W2V, BERT-PT) see only a stale view of the world — fine-tuning closes that gap. Details on the synthetic data design live in [`examples/data/README.md`](data/README.md); real CSMAR access is covered in [`docs/data.md`](../docs/data.md).

> **First time?** `jupyterlab` and `ipykernel` are demo-only extras and are **not** installed by a plain `uv sync`. Run once:
> ```bash
> uv sync --extra notebook
> uv run jupyter lab examples/quickstart.ipynb
> ```
> Prefer not to install Jupyter? Use the equivalent script: `uv run python examples/quickstart.py`.

In [ ]:
import json
import random
import shutil
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from gensim.models import Word2Vec

# Locate the project root by walking up to find pyproject.toml.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA = PROJECT_ROOT / "examples" / "data"
WORK = PROJECT_ROOT / "examples" / "_output"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def run_step(cmd: list[str], step: str) -> None:
    t0 = time.time()
    res = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        encoding="utf-8",
        errors="replace",
        env={**os.environ, "PYTHONUTF8": "1"},
        timeout=600,
    )
    print(f"[{step}] returncode={res.returncode}  ({time.time()-t0:.1f}s)")
    if res.returncode != 0:
        print(res.stderr[-1500:])
        raise RuntimeError(f"{step} failed")
    return res


print("Project root:", PROJECT_ROOT)
print("Data dir   :", DATA)
print("Workdir    :", WORK)

## Step 1 · Word2Vec on the pre-train period

Skip-gram on `portfolio_pretrained.csv`. Stocks that frequently co-occur in the same portfolios drift toward similar vectors — the "stocks as words" analogy. The output `.model` artifact doubles as BERT's tokenizer and initial embedding matrix in Step 2; the `*_embedding.csv` is the standalone Word2Vec embedding fed to RV in Step 4.

| Setting | Value |
|---|---|
| Algorithm | skip-gram with negative sampling |
| Embedding dim | 16 |
| Epochs | 10 |
| Context window | 5 |

In [ ]:
w2v_folder = WORK / "w2v"
w2v_name = "w2v_d16"

w2v_config = {
    "embedding_dim": 16,
    "epochs": 10,
    "window": 5,
    "min_count": 1,
    "sg": 1,
    "sample": 1e-3,
    "negative_sample": 3,
    "data_path": str(DATA / "portfolio_pretrained.csv"),
    "data_format": "csv",
    "workers": 1,
    "save_folder": str(w2v_folder),
    "save_name": w2v_name,
    "seed": SEED,
}
w2v_config_path = WORK / "w2v_config.json"
write_json(w2v_config_path, w2v_config)

run_step([sys.executable, "-m", "asset_embeddings.scripts.train.w2v", "-c", str(w2v_config_path)], "W2V")

w2v_model_path = w2v_folder / f"{w2v_name}.model"
w2v_embedding_csv = w2v_folder / f"{w2v_name}_embedding.csv"

# BERT vocab = W2V vocab + 3 special tokens ([MASK], [PAD], [UNK]).
w2v = Word2Vec.load(str(w2v_model_path))
BERT_VOCAB = len(w2v.wv) + 3
print(f"W2V vocab: {len(w2v.wv)}  ->  BERT vocab: {BERT_VOCAB}")

## Step 2 · BERT pretrain (MLM) on the pre-train period

Masked-language-model pretraining on the same PT portfolios. The W2V model from Step 1 supplies the vocabulary and warm-starts the embedding matrix; the transformer layers are randomly initialized.

| Setting | Value |
|---|---|
| Hidden dim | 16 (must match W2V) |
| Layers / heads | 2 / 2 |
| Intermediate size | 64 |
| Mask probability | 0.15 |
| Epochs | 8 |

Three CPU-friendly overrides vs. the production configs in `configs/pretrained/AssetBERT/`:

- `optimizer.optimizer_type: AdamW` — production uses `PagedAdam8bit`, which needs bitsandbytes + CUDA.
- `dataloader.num_workers: 0` — guards against Windows + Jupyter dataloader deadlocks.
- `train.mixed_precision: "no"` — explicit; AMP only helps on CUDA.

In [ ]:
bert_pt_folder = WORK / "bert_pt"
bert_pt_name = "bert_pt_d16"

bert_pt_config = {
    "model": {
        "model_type": "BERT",
        "model_checkpoint": None,
        "w2v_model": str(w2v_model_path),
        "embedding_file": None,
        "vocab_size": BERT_VOCAB,
        "hidden_size": 16,
        "num_hidden_layers": 2,
        "num_attention_heads": 2,
        "intermediate_size": 64,
        "max_position_embeddings": 64,
        "type_vocab_size": 1,
        "freeze_embedding": False,
        "freeze_encoder": False,
    },
    "tokenizer": {
        "w2v_model": str(w2v_model_path),
        "vocab_file": None,
        "pretrained_tokenizer_file": None,
        "alias_file": None,
    },
    "dataset": {
        "data_path": str(DATA / "portfolio_pretrained.csv"),
        "data_format": "csv",
        "id_key": "InvestorID",
        "portfolio_key": "Portfolio",
        "proportion1_key": "Proportion1",
        "proportion2_key": "Proportion2",
        "include_proportion": False,
        "max_length": 64,
        "num_repeats": 1,
        "mask_prob": 0.15,
        "mask_indices": None,
    },
    "dataloader": {
        "batch_size": 64,
        "shuffle": True,
        "num_workers": 0,
        "persistent_workers": False,
        "pin_memory": False,
        "drop_last": False,
    },
    "optimizer": {
        "optimizer_type": "AdamW",
        "optimizer_kwargs": None,
        "learning_rate": 0.001,
        "lr_scheduler_type": "cosine",
        "lr_scheduler_warmup_steps": 0,
        "lr_scheduler_train_steps": None,
        "lr_scheduler_num_cycles": 0.5,
        "lr_scheduler_power": 0,
    },
    "accelerator": {"mixed_precision": "no"},  # explicit; AMP only helps on CUDA
    "train": {
        "max_epochs": 8,
        "validation_split": 0.2,
        "split_method": "random",
        "best_metric": "val_loss",
        "clip_grad_norm": None,
        "clip_grad_value": None,
        "detect_anomaly": False,
        "log_every_steps": 200,
        "report_every_epochs": 1,
        "calculate_contextualized_embeddings": True,
        "save_every_epochs": 1,
        "save_folder": str(bert_pt_folder),
        "save_name": bert_pt_name,
        "save_format": ".safetensors",
        "seed": SEED,
    },
}
bert_pt_config_path = WORK / "bert_pt_config.json"
write_json(bert_pt_config_path, bert_pt_config)

run_step([sys.executable, "-m", "asset_embeddings.scripts.train.bert", "-c", str(bert_pt_config_path)], "BERT-PT")

bert_pt_checkpoint = bert_pt_folder / f"{bert_pt_name}_best" / "model.safetensors"
bert_pt_embedding_csv = bert_pt_folder / f"{bert_pt_name}_best_contextual_embedding.csv"
assert bert_pt_checkpoint.exists() and bert_pt_embedding_csv.exists()

## Step 3 · BERT fine-tune on the FT period

Warm-start from the BERT-PT checkpoint and continue MLM training on `portfolio_finetune.csv`. The FT period differs from PT by a cluster-level drift in latent space (see [`data/README.md`](data/README.md)) — fine-tuning is what teaches the model that drift direction.

Five things change vs. the PT config:

- `model.model_checkpoint` — point at the PT `.safetensors`.
- `dataset.data_path` — FT-period portfolios.
- `dataset.include_proportion: true` — proportions enter as an auxiliary signal.
- `optimizer.learning_rate: 5e-4` — gentler than the 1e-3 used for PT.
- `train.max_epochs: 5` and `dataloader.batch_size: 32` — shorter, calmer schedule.

In [ ]:
bert_ft_folder = WORK / "bert_ft"
bert_ft_name = "bert_ft_d16"

bert_ft_config = dict(bert_pt_config)
bert_ft_config["model"] = {**bert_pt_config["model"], "model_checkpoint": str(bert_pt_checkpoint)}
bert_ft_config["dataset"] = {
    **bert_pt_config["dataset"],
    "data_path": str(DATA / "portfolio_finetune.csv"),
    "include_proportion": True,
}
bert_ft_config["dataloader"] = {**bert_pt_config["dataloader"], "batch_size": 32}
bert_ft_config["optimizer"] = {**bert_pt_config["optimizer"], "learning_rate": 5e-4}
bert_ft_config["train"] = {
    **bert_pt_config["train"],
    "max_epochs": 5,
    "save_folder": str(bert_ft_folder),
    "save_name": bert_ft_name,
}

bert_ft_config_path = WORK / "bert_ft_config.json"
write_json(bert_ft_config_path, bert_ft_config)

run_step([sys.executable, "-m", "asset_embeddings.scripts.train.bert", "-c", str(bert_ft_config_path)], "BERT-FT")

bert_ft_embedding_csv = bert_ft_folder / f"{bert_ft_name}_best_contextual_embedding.csv"
assert bert_ft_embedding_csv.exists()

## Step 4 · Relative Valuation on all three embeddings

For each embedding file, the evaluator regresses log book equity on log market equity (controlling for size), then ridge-regresses the residual on the embedding. The reported R² measures how much cross-sectional variation in valuation is explained by the learned representation.

We run RV three times — once per embedding — and collect the scores into a comparison table.

In [ ]:
benchmark_dir = WORK / "benchmark"
benchmark_dir.mkdir(exist_ok=True)
shutil.copy(DATA / "benchmark.csv", benchmark_dir / "2024Q2.csv")


def rv_eval(embedding_csv: Path, model_type: str, label: str) -> float:
    cfg = {
        "model_type": model_type,
        "embedding_path": [str(embedding_csv)],
        "benchmark_path": str(benchmark_dir / "2024Q2.csv"),
        "periods": ["2024Q2"],
        "features": ["embedding"],
        "align_chars": True,
        "normalization": "minmax",
        "seed": SEED,
    }
    cfg_path = WORK / f"rv_{embedding_csv.stem}.json"
    write_json(cfg_path, cfg)
    res = subprocess.run(
        [sys.executable, "-m", "asset_embeddings.scripts.eval.rv", "-c", str(cfg_path)],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        encoding="utf-8",
        errors="replace",
        env={**os.environ, "PYTHONUTF8": "1"},
        timeout=120,
    )
    if res.returncode != 0:
        print(res.stderr[-1500:])
        raise RuntimeError(f"RV for {label} failed")
    for line in (res.stderr + res.stdout).splitlines():
        if "RV Score for period" in line:
            return float(line.rsplit(":", 1)[1].strip())
    raise RuntimeError(f"Could not extract RV score for {label}")


scores = {
    "W2V": rv_eval(w2v_embedding_csv, "AssetW2V", "W2V"),
    "BERT-PT": rv_eval(bert_pt_embedding_csv, "AssetBERT", "BERT-PT"),
    "BERT-FT": rv_eval(bert_ft_embedding_csv, "AssetBERT", "BERT-FT"),
}

baseline = scores["W2V"]
delta_ft = scores["BERT-FT"] - baseline
summary = pd.DataFrame(
    {
        "Trained on": ["PT data", "PT data", "FT data"],
        "RV @ 2024Q2": [f"{scores[k]:.4f}" for k in scores],
        "\u0394 vs W2V": ["baseline", f"{scores['BERT-PT']-baseline:+.4f}", f"{delta_ft:+.4f}  \u2190 FT lift"],
    },
    index=list(scores.keys()),
)
summary

## Results & interpretation

**W2V and BERT-PT plateau together** because they are both trained on the same PT-period portfolios. They recover the PT-period cluster structure, but the benchmark is FT-aligned and the FT period adds a cluster-level drift that PT-trained models cannot see.

**BERT-FT pulls ahead** because fine-tuning on FT-period portfolios lets it pick up that drift direction and align its representation with the FT benchmark. On this synthetic data the lift is around +0.07 R²; on real CSMAR data it is typically larger, since real markets carry richer time-varying structure (sector rotation, regime shifts, institutional crowding) than the cluster-drift toy model used here.

## Where to go next

- [`docs/training.md`](../docs/training.md) — full training pipeline on CSMAR data, all model variants
- [`docs/evaluation.md`](../docs/evaluation.md) — RV / RC / MP evaluation scripts and configs
- [`docs/data.md`](../docs/data.md) — CSMAR data acquisition and processing